# 4 — Zipf exponents and the lexical crossover

The central empirical claim of the paper: the rank-frequency curve of single
words is **not** a single power law. A Zipfian head with exponent close to 1
gives way, around a rank of order 10^4, to a steeper tail with exponent close to
2. The head is the optimized kernel lexicon; the tail is the peripheral lexicon.

Produces **Figure 1**, **Figure S2** and **Table S2**.

Reads `data_reduced/` only and finishes in seconds. Table S3 and the two
controls that show the tail exponent is not an artefact of corpus length or
vocabulary size are not here: they resample the raw token streams, so they live
in notebook 3 with everything else that needs `data_raw/`.

In [ ]:
import os
import subprocess
import sys

REPO = os.path.abspath("..") if os.path.isdir(os.path.join("..", "src")) else os.path.abspath(".")
sys.path.insert(0, os.path.join(REPO, "src"))

import plotting as P


def run(*command, must_succeed=True):
    """Run one pipeline step and, unlike a `!` cell, STOP if it fails.

    An IPython `!` cell throws away the exit status: a step that dies leaves no
    output, no error and no trace, and `nbconvert --execute` still reports the
    notebook as successful. Two defects in this pipeline's history hid exactly
    there, so every step below goes through this instead.

    `must_succeed=False` is used only for the two `--check` diagnostics of
    notebook 1, which print a loud banner rather than stopping the run.
    """
    print(">>", " ".join(str(c) for c in command), flush=True)
    code = subprocess.run([str(c) for c in command], check=False).returncode
    if code and must_succeed:
        raise RuntimeError(f"step failed with exit code {code} - read the output "
                           f"above; nothing after this point is valid")
    if code:
        rule = "*" * 72
        print(rule)
        print(f"*** THIS CHECK FAILED (exit code {code}). Read the output above")
        print("*** before going on: whatever depends on this corpus is missing")
        print("*** or wrong, and so is anything computed from it.")
        print(rule, flush=True)
    return code


def py(script, *args, must_succeed=True):
    """`run` for one of this repository's own scripts."""
    return run(sys.executable, os.path.join(REPO, "src", script), *args,
               must_succeed=must_succeed)


%matplotlib inline
USETEX = P.setup_style()
print("repo:", REPO, "| LaTeX text rendering:", USETEX)

## 4.1 The exponents — Table S2

Two estimators are reported for the same tail, because they use different
information and can therefore disagree:

* **two-regime OLS** on the rank-frequency curve — head over R ∈ [1, 10³], tail
  over R ≥ 10⁴ with count ≥ 2;
* the **discrete power-law MLE on the frequency spectrum**, P(f) ∝ f^(−β), with
  α = 1/(β−1). Reported at x_min = 2, so only the hapax legomena — the most
  undersampled bin — are dropped and no cutoff is tuned.

Every exponent carries a **fit-window band**: the spread over a set of a-priori
reasonable windows. That band, not the nominal OLS standard error, is the honest
error bar here. The standard error assumes independent residuals, which a smooth
log-log curve does not have, and it understates the uncertainty by about two
orders of magnitude.

The KS-optimal MLE of Clauset et al. is printed for contrast only: KS selects a
large cutoff, i.e. it fits the *head*, and returns α ≈ 1.

In [ ]:
py("fig1a_table.py")

## 4.2 Figure 1

Panel A is the five rank-frequency curves; panel B is the same text partitioned
into phrases (notebook 5 covers the construction and its exponents). Panel C of
the published figure is a typeset table of concept ranks, built from the numbers
notebook 5 produces.

In [ ]:
import figure_1

words = figure_1.word_curves()
phrases = figure_1.phrase_curves()
fig, _ = figure_1.compose(words, phrases)
for path in P.save_figure(fig, "fig1"):
    print("wrote", os.path.relpath(path, REPO))

## 4.3 Figure S2 — where the Zipf regime actually holds

A single fitted exponent cannot say *where* a curve is Zipfian. This figure
plots the local slope α(R), measured in a sliding window half a decade wide.
The phrase curves stay inside the band 1 ± 0.15 over most of their range; the
single-word curve leaves it early and climbs towards 2.

In [ ]:
import figure_SI2
from phrase_exponents import spectra

_, spec, _ = spectra(verbose=False)
fig = figure_SI2.draw(spec)
for path in P.save_figure(fig, "SI2"):
    print("wrote", os.path.relpath(path, REPO))

**Next:** `05_networks.ipynb`.